## Exploração dos Dados

### Importação de Bibliotecas Necessárias

In [ ]:
# Importar bibliotecas necessárias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

### Importação das Planilhas de Dados

In [ ]:
# Carregar os arquivos CSV
df_gf = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GF.csv")
df_gl = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GL.csv")
df_gm = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GM.csv")
df_gp = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GP.csv")
df_gt = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GT.csv")
df_gv = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GV.csv")
df_gw = pd.read_csv("/content/Grupo com registro entre 07-2024 a 06-2025- GW.csv")

### Divisão de Colunas em Múltiplas Colunas

In [ ]:
# Função para dividir colunas únicas em múltiplas colunas
def dividir_colunas(df):        
    # Extrair cabeçalho            
    cabecalho = df.columns[0].split('\t')   
    # Dividir dados
    dados = df.iloc[:, 0].str.split('\t', expand=True) 
    dados.columns = cabecalho
    return dados

# Aplicar a função a cada dataframe
df_gf = dividir_colunas(df_gf)
df_gl = dividir_colunas(df_gl)
df_gm = dividir_colunas(df_gm)
df_gp = dividir_colunas(df_gp)
df_gt = dividir_colunas(df_gt)
df_gv = dividir_colunas(df_gv)
df_gw = dividir_colunas(df_gw)

# Exibir cabeçalhos
display(df_gf.head())
display(df_gl.head())
display(df_gm.head())
display(df_gp.head())
display(df_gt.head())
display(df_gv.head())
display(df_gw.head())

### Conversão de Tipos de Dados

In [ ]:
# Definir colunas numéricas e categóricas
colunas_numericas = ['id_grupo', 'id_pagador', 'id_beneficiario', 'Numero_do_boleto', 'vl_boleto', 'vl_pagto', 'juros']
colunas_categoricas = ['status_boleto', 'pagador_cep', 'banco', 'tipo_juros', 'pagador_cidade', 'data_inclusao', 'data_vencto', 'dt_pagto']

# Função para converter tipos de colunas
def converter_tipos(df, colunas, tipo):
    for coluna in colunas:
        if coluna in df.columns:    
            # Substituir '\N' por NaN
            df[coluna] = df[coluna].replace('\\N', np.nan)  
            if tipo == 'float':
                # Converter para float
                df[coluna] = pd.to_numeric(df[coluna], errors='coerce')  
            elif tipo == 'datetime':
                # Converter para datetime
                df[coluna] = pd.to_datetime(df[coluna], errors='coerce')  
            else:
                # Converter para o tipo especificado
                df[coluna] = df[coluna].astype(tipo)

# Aplicar conversões
for df in [df_gf, df_gl, df_gm, df_gp, df_gt, df_gv, df_gw]:
    converter_tipos(df, colunas_numericas, 'float')
    converter_tipos(df, colunas_categoricas, 'category')

# Exibir informações dos dataframes
display(df_gf.info())
display(df_gl.info())
display(df_gm.info())
display(df_gp.info())
display(df_gt.info())
display(df_gv.info())
display(df_gw.info())

## Tratamento de Valores Ausentes

In [ ]:
# Remover colunas irrelevantes e preencher valores ausentes
colunas_para_remover_geral = [
    'tipo_multa',
    'multa'
]

for df in [df_gf, df_gl, df_gm, df_gp, df_gt, df_gv, df_gw]:
    # Remover colunas que existem no dataframe atual
    colunas_existentes_para_remover = [col for col in colunas_para_remover_geral if col in df.columns]
    df.drop(columns=colunas_existentes_para_remover, inplace=True)

    if 'vl_pagto' in df.columns:
        # Preencher valores ausentes com 0
        df['vl_pagto'] = df['vl_pagto'].fillna(0)  
    if 'pagador_cidade' in df.columns:
        # Garantir que a coluna seja categórica antes de adicionar a nova categoria
        df['pagador_cidade'] = df['pagador_cidade'].astype('category')
        df['pagador_cidade'] = df['pagador_cidade'].cat.add_categories('Desconhecido').fillna('Desconhecido')

    for coluna_data in ['data_inclusao', 'data_vencto', 'dt_pagto']:
        if coluna_data in df.columns:
            # Converter para datetime, substituindo '\N' por NaT e depois preenchendo NaT com NaT
            df[coluna_data] = pd.to_datetime(df[coluna_data].replace('\\N', np.nan), errors='coerce').fillna(pd.NaT)

## Estatística Descritiva dos Dados

In [ ]:
# Exibir estatísticas descritivas para cada dataframe
for df in [df_gf, df_gl, df_gm, df_gp, df_gt, df_gv, df_gw]:
    display(df.describe())

## Identificação e Tratamento de Outliers

In [ ]:
# Aplicar capping para tratar outliers
for df in [df_gf, df_gl, df_gm, df_gp, df_gt, df_gv, df_gw]:
    for coluna in colunas_numericas:
        if coluna in df.columns:
            # Calcular limites para capping
            Q1 = df[coluna].quantile(0.25)
            Q3 = df[coluna].quantile(0.75)
            IQR = Q3 - Q1
            limite_inferior = Q1 - 1.5 * IQR
            limite_superior = Q3 + 1.5 * IQR
            # Aplicar capping
            df[coluna] = np.where(df[coluna] < limite_inferior, limite_inferior, df[coluna])
            df[coluna] = np.where(df[coluna] > limite_superior, limite_superior, df[coluna])

## Codificação de Variáveis Categóricas

In [ ]:
# Aplicar one-hot encoding nas variáveis categóricas
for df in [df_gf, df_gl, df_gm, df_gp, df_gt, df_gv, df_gw]:
    colunas_para_codificar = [col for col in colunas_categoricas if col in df.columns]
    df = pd.get_dummies(df, columns=colunas_para_codificar, dummy_na=False)

## Normalização de Variáveis Numéricas

In [ ]:
# Normalizar colunas numéricas usando StandardScaler
for df in [df_gf, df_gl, df_gm, df_gp, df_gt, df_gv, df_gw]:
    scaler = StandardScaler()
    colunas_para_normalizar = [col for col in colunas_numericas if col in df.columns]
    if colunas_para_normalizar:
        df[colunas_para_normalizar] = scaler.fit_transform(df[colunas_para_normalizar])

## DataFrames Resultantes Após Normalização

In [ ]:
# Exibir o cabeçalho (head) de cada dataframe após a normalização
for df in [df_gf, df_gl, df_gm, df_gp, df_gt, df_gv, df_gw]:
    display(df.head())